### 6. Случайная величина имеет распределениe Парето


\begin{equation*} 
p(x) =
\begin{cases} 
\frac{\theta - 1}{x^{\theta}}, & \text{если } x \geq 1, \\
0, & \text{если } x < 1.
\end{cases}
\end{equation*} 



#### d)
Сгенерируйте выборку объема n = 100 для некоторого значения параметра $\theta$. Вычислите указанные выше доверительные интервалы для доверительной вероятности 0.95

In [36]:
import numpy as np
import random

theta = 5

def p(x):
    return (theta - 1) / x ** theta if x >= 0 else 0

def F(x):
    return (1 - x ** (1 - theta)) if x >= 0 else 0

def F_inv(y):
    return (1 - y) ** (1 / (1 - theta))

n = 100
array = [float(F_inv(random.random())) for _ in range(n)]

print(array)

[1.4234891229400977, 1.2958856444242515, 1.3688619651048741, 2.3902363607811736, 1.123946989385423, 1.108559601626475, 1.0337368913632512, 1.9050833010420414, 1.5522272178865508, 1.9058506204612924, 1.0955665533927228, 1.0143234726506947, 2.191529318694888, 1.2194046760632486, 2.8214522225858434, 1.0009243450124825, 1.0473473901911663, 1.96168206561703, 1.0072225145463995, 1.1123233563557045, 1.0553697278203573, 1.0005884433795116, 1.0908913026822025, 1.6814296848760384, 1.9429848740344753, 1.0505056451387038, 1.0417916229208322, 1.371603099697624, 1.1540999608355629, 1.579369789902145, 1.1127292726068414, 1.3545652235911372, 1.2192890990623817, 1.0135895480353538, 1.0903706610518857, 1.0040128036614115, 1.745695211157247, 1.1061211907159263, 1.1009977335856083, 1.0136934280895482, 1.1214774581422284, 1.1802997480301063, 1.4597171810728422, 1.062109983411646, 1.0749850740440465, 1.997153911867097, 1.195347568351543, 1.7445550897263322, 1.0102302928691635, 1.5402011724807207, 1.21111230

$$ \tilde{\theta} = 1 + \frac{1}{\overline{ln(x)}} $$ $$ P(\tilde{\theta} - \frac{(\tilde{\theta} - 1)t_2}{\sqrt{n}} < \theta < \tilde{\theta} - \frac{(\tilde{\theta} - 1)t_1}{\sqrt{n}}) = \beta $$

In [37]:
t_1 = -1.96
t_2 = 1.96

beta = 0.95

theta_with_wave = 1 + 1 / np.mean([np.log(x) for x in array])

left = theta_with_wave - ((theta_with_wave - 1) * t_2) / np.sqrt(n)
right = theta_with_wave -  ((theta_with_wave - 1) * t_1) / np.sqrt(n)

print(f'Доверительный интервал: ({left}, {right})')
print(f'Длина интервала: {right - left}')


Доверительный интервал: (4.288536126055534, 5.891901998460718)
Длина интервала: 1.6033658724051847


### Доверительный интервал для медианы

$$
P(g(\tilde{\theta}) - \frac{t_2 \cdot ln2 \cdot 2 ^ {\frac{1}{(\tilde{\theta} - 1)}}}{(\tilde{\theta} - 1)\sqrt{n}} < g(\theta) < g(\tilde{\theta}) - \frac{t_1 \cdot ln2 \cdot 2 ^ {\frac{1}{(\tilde{\theta} - 1)}}}{(\tilde{\theta} - 1)\sqrt{n}}) = \beta
$$

In [38]:

def g(theta : float) -> float:
    return 2 ** (1 / (theta - 1))

left = g(theta_with_wave) - (t_2 * np.log(2) * g(theta_with_wave)) / ((theta_with_wave - 1) * np.sqrt(n)) 
right = g(theta_with_wave) - (t_1 * np.log(2) * g(theta_with_wave)) / ((theta_with_wave - 1) * np.sqrt(n)) 

median = (sorted(array)[49] + sorted(array)[50]) / 2

print("Медиана: ", median)

print(f'Доверительный интервал: ({left}, {right})')
print(f'Длина интервала: {right - left}')



Медиана:  1.1614111564820324
Доверительный интервал: (1.1453214733579233, 1.2240192537651586)
Длина интервала: 0.0786977804072353


### t) Численно постройте бутстраповский доверительный интервал двумя способами, используя параметрический бутстрап и непараметрический бутстрап.

In [39]:
# НЕПАРАМЕТРИЧЕСКИЙ
bootstrap_iteration = 1000

theta_with_wave = 1 + 1 / np.mean([np.log(x) for x in array]) 

bootstrap_delta = []

for i in range(bootstrap_iteration):
    bootstrap_array = np.random.choice(array, size=len(array), replace=True)
    bootstrap_theta = 1 + 1 / np.mean([np.log(x) for x in bootstrap_array])
    bootstrap_delta.append(bootstrap_theta - theta_with_wave)

bootstrap_delta.sort()

t_1 = bootstrap_delta[int(bootstrap_iteration * (1 - beta) / 2)]
t_2 = bootstrap_delta[int(bootstrap_iteration * (1 + beta) / 2)]

right = -(t_1 - theta_with_wave)
left = -(t_2 - theta_with_wave)

print("\tНЕПАРАМЕТРИЧЕСКИЙ")
print(f'Доверительный интервал: ({left}, {right})')
print(f'Длина интервала: {right - left}')


# ПАРАМЕТРИЧЕСКИЙ
bootstrap_iteration = 10000

def F_inv_star(y: float, theta_w: float) -> float:
    return (1 - y) ** (1 / (1 - theta_w))

for i in range(bootstrap_iteration):
    bootstrap_array = [F_inv_star(random.random(), theta_with_wave) for _ in range(n)]
    bootstrap_theta = 1 + 1 / np.mean([np.log(x) for x in bootstrap_array])
    bootstrap_delta.append(bootstrap_theta - theta_with_wave)

bootstrap_delta.sort()
t_1 = bootstrap_delta[int(bootstrap_iteration * (1 - beta) / 2)]
t_2 = bootstrap_delta[int(bootstrap_iteration * (1 + beta) / 2)]
right = -(t_1 - theta_with_wave)
left = -(t_2 - theta_with_wave)

print("\n\tПАРАМЕТРИЧЕСКИЙ")
print(f'Доверительный интервал: ({left}, {right})')
print(f'Длина интервала: {right - left}')

	НЕПАРАМЕТРИЧЕСКИЙ
Доверительный интервал: (4.137597308806593, 5.774113010035419)
Длина интервала: 1.6365157012288263

	ПАРАМЕТРИЧЕСКИЙ
Доверительный интервал: (4.5397272220449745, 5.799804526029305)
Длина интервала: 1.260077303984331


### f) Сравнить все интервалы.


### параметрический бутстрап < асимптотический метод < непараметрический бутстрап